# Graph Construction Demo

Build a brain graph from supervoxels.

**Eq. 7 (Edge weights):** w_ij = alpha*sim_intensity + beta*sim_atlas + gamma*sim_spatial + delta*P_co_occurrence

**Node features** (~55 dims):
- Per-modality stats: mean, std, min, max (4 modalities x 4 = 16)
- Centroid: x, y, z (3)
- Volume: voxel count (1)
- Atlas one-hot: ArterialAtlas136 (32)
- Connectivity probabilities: [p0, p1, p2] (3)

In [ ]:
%matplotlib inline

import sys
sys.path.insert(0, "../src")

import numpy as np
import matplotlib.pyplot as plt
import torch

from stroke_gat.config import load_config
from stroke_gat.data.service import DataService
from stroke_gat.graph.builder import GraphBuilder
from stroke_gat.visualization.graph_3d import plot_graph_3d_interactive, plot_graph_2d_projection

print("Imports successful.")

In [ ]:
# Load config and prepare data via DataService
config = load_config("../configs/default.yaml")
service = DataService(config.paths)

subjects = service.discover_subjects_bids()
subject_id = subjects[0]
print(f"Building graph for subject: {subject_id}")

# Preprocess: load, 4D collapse, normalize, co-register to T1
modality_data, masks, atlas_aligned = service.preprocess_subject(subject_id)
metadata = DataService.get_subject_metadata(subject_id, masks)

print(f"Modalities: {list(modality_data.keys())}")
print(f"Masks: {list(masks.keys())}")
print(f"Metadata: {metadata}")

In [ ]:
# Build the graph using GraphBuilder
builder = GraphBuilder(slic_config=config.slic, graph_config=config.graph)

graph, supervoxel_labels = builder.build(
    modalities=modality_data,
    masks=masks,
    atlas=atlas_aligned,
    subject_id=subject_id,
    metadata=metadata,
)

print(f"\nGraph Statistics for Subject {subject_id}")
print("=" * 50)
print(f"Number of nodes:     {graph.num_nodes:>8,}")
print(f"Number of edges:     {graph.num_edges:>8,}")
print(f"Node feature dim:    {graph.x.shape[1]:>8}")
print(f"Average degree:      {graph.num_edges / graph.num_nodes:>8.1f}")

if hasattr(graph, 'edge_weight') and graph.edge_weight is not None:
    ew = graph.edge_weight.numpy()
    print(f"\nEdge Weight Statistics:")
    print(f"  Min:  {ew.min():.4f}  Max: {ew.max():.4f}  Mean: {ew.mean():.4f}")

# Label distribution
labels = graph.y.numpy()
print(f"\nNode Label Distribution:")
for c, name in enumerate(["No Lesion", "Acute", "Chronic"]):
    count = (labels == c).sum()
    pct = 100.0 * count / len(labels)
    print(f"  {name:>12s}: {count:>6,} nodes ({pct:.1f}%)")

In [ ]:
# Node feature distributions (intensity stats: 4 modalities x 4 stats = 16)
features = graph.x.numpy()
mod_names = ["T1", "FLAIR", "ADC", "TRACE"]
stat_names = ["Mean", "Std", "Min", "Max"]

fig, axes = plt.subplots(4, 4, figsize=(16, 12))
for i, mod in enumerate(mod_names):
    for j, stat in enumerate(stat_names):
        feat_idx = i * 4 + j
        axes[i, j].hist(features[:, feat_idx], bins=40, color="steelblue",
                        edgecolor="white", alpha=0.8)
        axes[i, j].set_title(f"{mod} - {stat}", fontsize=10)
        axes[i, j].tick_params(labelsize=8)

fig.suptitle("Node Feature Distributions (Intensity Statistics)",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# Edge weight and degree distributions
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

if hasattr(graph, 'edge_weight') and graph.edge_weight is not None:
    ew = graph.edge_weight.numpy()
    axes[0].hist(ew, bins=60, color="coral", edgecolor="white", alpha=0.85)
    axes[0].set_xlabel("Edge Weight", fontsize=12)
    axes[0].set_ylabel("Count", fontsize=12)
    axes[0].set_title("Edge Weight Distribution", fontsize=13)

from torch_geometric.utils import degree
deg = degree(graph.edge_index[0], num_nodes=graph.num_nodes).numpy()
axes[1].hist(deg, bins=30, color="mediumpurple", edgecolor="white", alpha=0.85)
axes[1].set_xlabel("Node Degree", fontsize=12)
axes[1].set_ylabel("Count", fontsize=12)
axes[1].set_title(f"Degree Distribution (mean={deg.mean():.1f})", fontsize=13)

plt.tight_layout()
plt.show()

In [ ]:
# Interactive 3D graph visualization (nodes colored by label)
fig_3d = plot_graph_3d_interactive(
    graph,
    title=f"Brain Graph - Subject {subject_id}",
    node_size=3.0,
    edge_opacity=0.1,
)
fig_3d.show()

In [ ]:
# 2D projection and t-SNE
fig = plot_graph_2d_projection(
    graph,
    projection="axial",
    title=f"Graph Axial Projection - Subject {subject_id}",
    node_size=5.0,
)
plt.show()

# t-SNE of node features
from sklearn.manifold import TSNE

tsne = TSNE(n_components=2, perplexity=30, random_state=42)
embedded = tsne.fit_transform(features)

colors = ["#2196F3", "#F44336", "#FF9800"]
class_names = ["No Lesion", "Acute", "Chronic"]

fig, ax = plt.subplots(figsize=(10, 8))
for c in range(3):
    mask = labels == c
    ax.scatter(embedded[mask, 0], embedded[mask, 1], s=5, c=colors[c],
               label=class_names[c], alpha=0.6, edgecolors="none")
ax.set_title("t-SNE of Node Features", fontsize=13)
ax.legend(markerscale=5)
plt.tight_layout()
plt.show()

## Notes

- Graph has ~8000 nodes with ~55-dim feature vectors
- 26-connectivity yields ~16-26 edges per node
- Edge weights encode 4 components: intensity sim, atlas sim, spatial proximity, co-occurrence
- t-SNE shows partial class separability, motivating GAT message passing

Next: See `04_training_demo.ipynb` for training the GAT model.